In [1]:
# Import packages
import os
from pathlib import Path
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold

In [2]:
# Load in dataset
path = Path("../../../../local3/sswee/music_download/physionet.org/files/music-sudden-cardiac-death/1.0.1/subject-info.csv")

df = pd.read_csv(
    path,
    sep=";",          # correct delimiter
    decimal=",",      # European decimal format
    engine="python",  # handle irregular formatting
    na_values=["", "NA"] # handles missing values
)

# Clean (tabs inside numbers)
df = df.replace(r"\t", ".", regex=True)

# Convert age to numeric
df["Age"] = (
    df["Age"]
    .astype(str)
    .str.strip()
    .str.replace(",", ".", regex=False)
)
df["Age"] = pd.to_numeric(df["Age"], errors="coerce")

print(df.shape) # 992 patients with 103 columns

df.head()

(992, 103)


,Patient ID,Follow-up period from enrollment (days),days_4years,Exit of the study,Cause of death,Age,Gender (male=1),Weight (kg),Height (cm),Body Mass Index (Kg/m2),...,Angiotensin-II receptor blocker (yes=1),Anticoagulants/antitrombotics (yes=1),Betablockers (yes=1),Digoxin (yes=1),Loop diuretics (yes=1),Spironolactone (yes=1),Statins (yes=1),Hidralazina (yes=1),ACE inhibitor (yes=1),Nitrovasodilator (yes=1)
0,P0001,2065,1460,NaN,0,58.0,1,83,163,31.2,...,0,1,1,1,1,0,0,0,1,0
1,P0002,2045,1460,NaN,0,58.0,1,74,160,28.9,...,1,1,1,0,0,0,1,0,0,0
2,P0003,2044,1460,NaN,0,69.0,1,83,174,27.4,...,1,1,1,1,1,0,0,0,0,0
3,P0004,2044,1460,NaN,0,56.0,0,84,165,30.9,...,1,1,1,0,1,1,0,0,0,0
4,P0005,2043,1460,NaN,0,70.0,1,97,183,29.0,...,0,1,1,0,1,0,1,0,1,1


In [3]:
df["Patient ID"] = df["Patient ID"].str.replace("P", "", regex=False)
df["Patient ID"] = df["Patient ID"].str.strip()

In [4]:
# Patient selection
mask = (
    (df["Holter available"] != 0) &  # Select patients with Holter
    (df["Cause of death"].isin([0, 3, 6, 7])) &  # Known survival or cardiac death
    (df["Prior implantable device"] == 0) &  # Remove pacemaker patients
    ((df["Exit of the study"].ne(2)) | (df["Exit of the study"].isna()))  # Remove cardiac transplant
)

df2 = df[mask].copy()

# Combine Cause of death codes:
# 6 and 7 -> Pump failure death
df2.loc[df2["Cause of death"].isin([6, 7]), "Cause of death"] = 6

# Table of number of patients per outcome
cause_counts = df2["Cause of death"].value_counts().sort_index()
print(cause_counts)

# Note: Original paper reports 94/996 SCDs (9.4%) and 111/996 PFDs (11.1%)
# Final counts align very well with 74/746 SCDs (9.9%) and 87/746 PFds (11.7%)

Cause of death
0    585
3     74
6     87
Name: count, dtype: int64


In [5]:
# Select Patient ID and Cause of death columns (labels for training/validating/testing)
labels = df2[["Patient ID", "Cause of death"]].copy()
labels = labels.rename(columns={"Cause of death": "label"})

# Reset index so it matches positional indices
labels = labels.reset_index(drop=True)

labels["label"].value_counts()

# Stratified K-fold
skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

labels["fold"] = -1

for fold, (_, val_idx) in enumerate(
    skf.split(labels["Patient ID"], labels["label"])
):
    labels.loc[val_idx, "fold"] = fold

# Check counts in each fold
labels["fold"].value_counts()
labels.groupby(["fold", "label"]).size().unstack()

label,0,3,6
fold,,,
0,117,15,18
1,117,15,17
2,117,15,17
3,117,15,17
4,117,14,18


In [6]:
# Save labels
path = Path("../../../../local3/sswee/music_download/physionet.org/files/music-sudden-cardiac-death/1.0.1")
out_file = path / "music_patient_folds_5cv.csv"
labels.to_csv(out_file, index=False)

In [7]:
# Smoke test (test on 30 patients, 10 from each group)
# Reproducibility
RANDOM_STATE = 42

# Sanity check
assert set(labels["label"].unique()) == {0, 3, 6}

# Sample 10 patients from each class
sampled = (
    labels
    .groupby("label", group_keys=False)
    .apply(lambda x: x.sample(n=10, random_state=RANDOM_STATE))
    .reset_index(drop=True)
)

print(sampled["label"].value_counts())

path = Path("../../../../local3/sswee/music_download/physionet.org/files/music-sudden-cardiac-death/1.0.1")

out_file = path / "music_patient_test30.csv"
sampled.to_csv(out_file, index=False)

print(f"Saved test cohort to: {out_file.resolve()}")


label
0    10
3    10
6    10
Name: count, dtype: int64
Saved test cohort to: /local3/sswee/music_download/physionet.org/files/music-sudden-cardiac-death/1.0.1/music_patient_test30.csv


/tmp/ipykernel_1684808/4139209444.py:12: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.sample(n=10, random_state=RANDOM_STATE))


In [8]:
sampled.head()

,Patient ID,label,fold
0,0693,0,2
1,0749,0,3
2,0168,0,1
3,0828,0,3
4,0912,0,2


In [9]:
labels.head()

,Patient ID,label,fold
0,0001,0,3
1,0002,0,3
2,0003,0,0
3,0004,0,4
4,0006,0,4


In [11]:
# Add ECG Impressions to dataframe
impressions = ["Ventricular Extrasystole", "Ventricular Tachycardia", "Non-sustained ventricular tachycardia (CH>10)", 
               "Paroxysmal supraventricular tachyarrhythmia", "Bradycardia"]

# Create ECG reports
class ECGReport:
    def __init__(self, ventricular_extrasystole, ventricular_tachycardia, non_sustained_ventricular_tachycardia, paroxysmal_supraventricular_tachyarrhythmia, bradycardia):
        self.ventricular_extrasystole = ventricular_extrasystole
        self.ventricular_tachycardia = ventricular_tachycardia
        self.non_sustained_ventricular_tachycardia = non_sustained_ventricular_tachycardia
        self.paroxysmal_supraventricular_tachyarrhythmia = paroxysmal_supraventricular_tachyarrhythmia
        self.bradycardia = bradycardia
        
    def interpret_ventricular_extrasystole(self):
        ventricular_extrasystole_dict = {
            0: "No",
            1: "Monomorphic",
            2: "Polymorphic",
            3: "Couplets"
        }
        return ventricular_extrasystole_dict.get(self.ventricular_extrasystole, "Unknown ventricular extrasystole code")
    
    def interpret_ventricular_tachycardia(self):
        ventricular_tachycardia_dict = {
            0: "No",
            1: "Non-sustained VT",
            2: "Sustained VT", 
            3: "Torsade de Points"
        }
        return ventricular_tachycardia_dict.get(self.ventricular_tachycardia, "Unknown ventricular tachycardia code")
    
    def interpret_non_sustained_ventricular_tachycardia(self):
        non_sustained_ventricular_tachycardia_dict = {
            0: "No",
            1: "Yes"
        }
        return non_sustained_ventricular_tachycardia_dict.get(self.non_sustained_ventricular_tachycardia, "Unknown non sustained ventricular tachycardia code")
    
    def interpret_paroxysmal_supraventricular_tachyarrhythmia(self):
        paroxysmal_supraventricular_tachyarrhythmia_dict = {
            0: "No", 
            1: "TPSV", 
            2: "Parosysmal AF", 
            3: "Paroxismal flutter", 
            4: "Others"
        }
        return paroxysmal_supraventricular_tachyarrhythmia_dict.get(self.paroxysmal_supraventricular_tachyarrhythmia, "Unknown paroxysmal supraventricular tachyarrhythmia code")
    
    def interpret_bradycardia(self):
        bradycardia_dict = {
            0: "No",
            1: "Sinus Node Dysfunction", 
            2: "First-degree Atrioventricular block (AVB)",
            3: "Second-degree AVB - type I",
            4: "Second-degree AVB - type II", 
            5: "Third-degree AVB", 
            6: "Paroxysmal AVB"
        }
        return bradycardia_dict.get(self.bradycardia, "Unknown bradycardia code")
    
    def generate_report(self):
        return f"""
        - Ventricular Extrasystole: {self.interpret_ventricular_extrasystole()}
        - Ventricular Tachycardia: {self.interpret_ventricular_tachycardia()}
        - Non-sustained ventricular tachycardia (CH>10): {self.interpret_non_sustained_ventricular_tachycardia()}
        - Paroxysmal supraventricular tachyarrhythmia: {self.interpret_paroxysmal_supraventricular_tachyarrhythmia()}
        - Bradycardia: {self.interpret_bradycardia()}
            """

# Generate ECG impressions for all patients

def safe_int(x):
    if pd.isna(x):
        return None
    return int(x)

df2['ECG_impressions'] = df2.apply(
    lambda row: ECGReport(
        safe_int(row["Ventricular Extrasystole"]),
        safe_int(row["Ventricular Tachycardia"]),
        safe_int(row["Non-sustained ventricular tachycardia (CH>10)"]),
        safe_int(row["Paroxysmal supraventricular tachyarrhythmia"]),
        safe_int(row["Bradycardia"])
    ).generate_report(),
    axis=1
)

# # Generate ECG impressions for all patients
# df2['ECG_impressions'] = df2.apply(lambda row: ECGReport(
#     row["Ventricular Extrasystole"], row["Ventricular Tachycardia"], row["Non-sustained ventricular tachycardia (CH>10)"], row["Paroxysmal supraventricular tachyarrhythmia"], row["Bradycardia"]).generate_report(), axis = 1)

In [12]:
df2.columns

Index(['Patient ID', 'Follow-up period from enrollment (days)', 'days_4years',
       'Exit of the study', 'Cause of death', 'Age', 'Gender (male=1)',
       'Weight (kg)', 'Height (cm)', 'Body Mass Index (Kg/m2)',
       ...
       'Anticoagulants/antitrombotics  (yes=1)', 'Betablockers (yes=1)',
       'Digoxin (yes=1)', 'Loop diuretics (yes=1)', 'Spironolactone (yes=1)',
       'Statins (yes=1)', 'Hidralazina (yes=1)', 'ACE inhibitor (yes=1)',
       'Nitrovasodilator (yes=1)', 'ECG_impressions'],
      dtype='object', length=104)

In [13]:
df2.shape

(746, 104)

In [17]:
# Test dictionary 
def generate_dictionary(row):
    # Create a dictionary to store non-missing values
    patient_data = {col: row[col] for col in df2.columns if pd.notna(row[col])}
    return patient_data
generate_dictionary(df2.iloc[745])

{'Patient ID': '1073',
 'Follow-up period from enrollment (days)': 1365,
 'days_4years': 1365,
 'Cause of death': 0,
 'Age': 70.0,
 'Gender (male=1)': 0,
 'Weight (kg)': 63,
 'Height (cm)': 144,
 'Body Mass Index (Kg/m2)': 30.4,
 'NYHA class': 2,
 'Diastolic blood  pressure (mmHg)': 86,
 'Systolic blood pressure (mmHg)': 134,
 'HF etiology - Diagnosis': 1,
 'Diabetes (yes=1)': 1,
 'History of dyslipemia (yes=1)': 0,
 'Peripheral vascular disease (yes=1)': 0,
 'History of hypertension (yes=1)': 0,
 'Prior Myocardial Infarction (yes=1)': 0,
 'Prior implantable device': 0,
 'Prior Revascularization': 0,
 'Syncope': 1,
 'daily smoking (cigarretes/day)': 0,
 'smoke-free time (years)': 0,
 'cigarettes /year': 0,
 'alcohol consumption (standard units)': 0,
 'Albumin (g/L)': 44.0,
 'ALT or GPT (IU/L)': 12.0,
 'AST or GOT (IU/L)': 13.0,
 'Normalized Troponin': 0.5,
 'Total Cholesterol (mmol/L)': 6.21,
 'Creatinine (?mol/L)': 80.0,
 'Gamma-glutamil transpeptidase (IU/L)': 89.0,
 'Glucose (mmol/L

In [19]:
# Test prompt 
def generate_prompt(row):
    # Create a dictionary to store non-missing values
    patient_data = {col: row[col] for col in df2.columns if pd.notna(row[col])}

    # Start the prompt
    # prompt = "Generate a structured clinical note based on the following data:\n\n"
    prompt = ""
    
    # Add demographic information 
    if "Age" in patient_data:
        prompt += f"Age: {patient_data['Age']}\n"
    if "Gender (male=1)" in patient_data:
        if patient_data['Gender (male=1)'] == 1:
            prompt += f"Gender: Male \n"
        elif patient_data['Gender (male=1)'] == 0:
            prompt += f"Gender: Female \n"
    if "Weight (kg)" in patient_data:
        prompt += f"Weight: {patient_data['Weight (kg)']} kg\n"
    if "Height (cm)" in patient_data:
        prompt += f"Height: {patient_data['Height (cm)']} cm\n"

    # Add clinical features
    if "NYHA class" in patient_data:
        if patient_data['NYHA class'] == 2:
            prompt += f"NYHA Class: II\n"
        elif patient_data['NYHA class'] == 3:
            prompt += f"NYHA Class: III\n"
    if ("Systolic blood pressure (mmHg)" in patient_data) and ("Diastolic blood  pressure (mmHg)" in patient_data):
        prompt += f"Blood Pressure: {patient_data['Systolic blood pressure (mmHg)']}/{patient_data['Diastolic blood  pressure (mmHg)']} mmHg\n"

    # Past medical history
    past_medical_conditions = []
    for condition in ["HF etiology - Diagnosis", "Diabetes (yes=1)", "History of dyslipemia (yes=1)", "Peripheral vascular disease (yes=1)",
                     "History of hypertension (yes=1)", "Prior Myocardial Infarction (yes=1)"]:
        if (condition in patient_data) and (condition == "HF etiology - Diagnosis"):
            if patient_data['HF etiology - Diagnosis'] == 1:
                past_medical_conditions.append("Idiopathic dilated cardiomyopathy")
                # prompt += f"HF Etiology: Idiopathic dilated cardiomyopathy\n"
            if patient_data['HF etiology - Diagnosis'] == 2:
                past_medical_conditions.append("Ischemic dilated cardiomyopathy")
                # prompt += f"HF Etiology: Ischemic dilated cardiomyopathy\n"
            if patient_data['HF etiology - Diagnosis'] == 3:
                past_medical_conditions.append("Enolic dilated cardiomyopathy")
                # prompt += f"HF Etiology: Enolic dilated cardiomyopathy\n"
            if patient_data['HF etiology - Diagnosis'] == 4:
                past_medical_conditions.append("Valvular cardiomyopathy")
                # prompt += f"HF Etiology: Valvular cardiomyopathy\n"
            if patient_data['HF etiology - Diagnosis'] == 5:
                past_medical_conditions.append("Toxic dilated cardiomyopathy")
                #prompt += f"HF Etiology: Toxic dilated cardiomyopathy\n"
            if patient_data['HF etiology - Diagnosis'] == 6:
                past_medical_conditions.append("Post-myocardial dilated cardiomyopathy")
                # prompt += f"HF Etiology: Post-myocardial dilated cardiomyopathy\n"
            if patient_data['HF etiology - Diagnosis'] == 7:
                past_medical_conditions.append("Hypertropic cardiomyopathy")
                # prompt += f"HF Etiology: Hypertropic cardiomyopathy\n"
            if patient_data['HF etiology - Diagnosis'] == 8:
                past_medical_conditions.append("Hypertensive cardiomyopathy")
                # prompt += f"HF Etiology: Hypertensive cardiomyopathy\n"
            if patient_data['HF etiology - Diagnosis'] == 9:
                past_medical_conditions.append("Other HF etiology")
                # prompt += f"HF Etiology: Other\n"
        elif (condition in patient_data) and (condition == "Diabetes (yes=1)"):
            if patient_data['Diabetes (yes=1)'] == 1:
                past_medical_conditions.append("Diabetes")
        elif (condition in patient_data) and (condition == "History of dyslipemia (yes=1)"):
            if patient_data['History of dyslipemia (yes=1)'] == 1:
                past_medical_conditions.append("Dyslipemia")
        elif (condition in patient_data) and (condition == "Peripheral vascular disease (yes=1)"):
            if patient_data['Peripheral vascular disease (yes=1)'] == 1:
                past_medical_conditions.append("Peripheral vascular disease")
        elif (condition in patient_data) and (condition == "History of hypertension (yes=1)"):
            if patient_data['History of hypertension (yes=1)'] == 1:
                past_medical_conditions.append("Hypertension")
        elif (condition in patient_data) and (condition == "Prior Myocardial Infarction (yes=1)"):
            if patient_data['Prior Myocardial Infarction (yes=1)'] == 1:
                past_medical_conditions.append("Myocardial Infarction")
    if past_medical_conditions:
        prompt += "Past Medical History: " + ", ".join(past_medical_conditions) + "\n"
    else:
        prompt += "Past Medical History: None reported.\n"

    # Lab results
    lab_tests = ['Albumin (g/L)', 'ALT or GPT (IU/L)', 'AST or GOT (IU/L)', 'Total Cholesterol (mmol/L)', 'Creatinine (?mol/L)',
                 'Gamma-glutamil transpeptidase (IU/L)', 'Glucose (mmol/L)', 'Hemoglobin (g/L)', 'HDL (mmol/L)', 
                 'Potassium (mEq/L)', 'LDL (mmol/L)', 'Sodium (mEq/L)', 'Pro-BNP (ng/L)',  'Protein (g/L)', 'T3 (pg/dL)', 
                 'T4 (ng/L)', 'Troponin (ng/mL)', 'TSH (mIU/L)', 'Urea (mg/dL)']
    for test in lab_tests:
        # Special case for units of creatining
        if (test in patient_data) and (test == 'Creatinine (?mol/L)'):
            prompt += f"Creatinine (mmol/L): {patient_data[test]}\n"
        elif test in patient_data:
            prompt += f"{test}: {patient_data[test]}\n"
    
    # LVEF
    if "LVEF (%)" in patient_data:
        # prompt += f"LVEF (%): {patient_data["LVEF (%)"]}\n"
        prompt += f"LVEF (%): {patient_data['LVEF (%)']}\n"
        
    # Medication
    current_medications = []
    medications = {
    'Calcium channel blocker (yes=1)': "Calcium Channel Blocker",
    'Diabetes medication (yes=1)': "Diabetes Medication",
    'Amiodarone (yes=1)': "Amiodarone",
    'Angiotensin-II receptor blocker (yes=1)': "Angiotensin II Receptor Blocker",
    'Anticoagulants/antitrombotics (yes=1)': "Anticoagulants/Antithrombotics",
    'Betablockers (yes=1)': "Beta Blockers",
    'Digoxin (yes=1)': "Digoxin",
    'Loop diuretics (yes=1)': "Loop Diuretics",
    'Spironolactone (yes=1)': "Spironolactone",
    'Statins (yes=1)': "Statins",
    'Hidralazina (yes=1)': "Hydralazine",
    'ACE inhibitor (yes=1)': "ACE Inhibitor",
    'Nitrovasodilator (yes=1)': "Nitrovasodilator"
    }
    for key, value in medications.items():
        if (key in patient_data) and (patient_data[key]) == 1:
            current_medications.append(value)
    if current_medications:
        prompt += "Medications: " + ", ".join(current_medications) + "\n"
    else:
        prompt += "Medications: None reported.\n"
    
    # Holter ECG Features (Impressions only)
    if 'ECG_impressions' in patient_data and isinstance(patient_data['ECG_impressions'], str):
        prompt += "ECG Impression:\n"
        prompt += patient_data['ECG_impressions'].strip() + "\n"
    else:
        prompt += "ECG Impression: Not reported.\n"

    return prompt
    
print(generate_prompt(df2.iloc[745]))

Age: 70.0
Gender: Female 
Weight: 63 kg
Height: 144 cm
NYHA Class: II
Blood Pressure: 134/86 mmHg
Past Medical History: Idiopathic dilated cardiomyopathy, Diabetes
Albumin (g/L): 44.0
ALT or GPT (IU/L): 12.0
AST or GOT (IU/L): 13.0
Total Cholesterol (mmol/L): 6.21
Creatinine (mmol/L): 80.0
Gamma-glutamil transpeptidase (IU/L): 89.0
Glucose (mmol/L): 5.5
Hemoglobin (g/L): 145.0
HDL (mmol/L): 1.68
Potassium (mEq/L): 4.5
LDL (mmol/L): 3.23
Sodium (mEq/L): 141.0
Protein (g/L): 73.0
T3 (pg/dL): 0.05
T4 (ng/L): 12.0
Troponin (ng/mL): 0.03
TSH (mIU/L): 1.43
Urea (mg/dL): 8.82
LVEF (%): 40.0
Medications: Beta Blockers, Loop Diuretics, ACE Inhibitor
ECG Impression:
- Ventricular Extrasystole: Monomorphic
        - Ventricular Tachycardia: No
        - Non-sustained ventricular tachycardia (CH>10): No
        - Paroxysmal supraventricular tachyarrhythmia: Unknown paroxysmal supraventricular tachyarrhythmia code
        - Bradycardia: Unknown bradycardia code



In [40]:
# Create prompt dataframe
df_prompts = df2[['Patient ID']].copy().reset_index(drop = True)
df_prompts['Prompts'] = None
for i in range(len(df_prompts)):
    df_prompts.loc[i, 'Prompts'] = generate_prompt(df2.iloc[i])

In [41]:
df_prompts.shape

(746, 2)

In [42]:
df_prompts.shape

(746, 2)

In [43]:
print(df_prompts.iloc[0,1])

Age: 58.0
Gender: Male 
Weight: 83 kg
Height: 163 cm
NYHA Class: III
Blood Pressure: 110/75 mmHg
Past Medical History: Idiopathic dilated cardiomyopathy
Albumin (g/L): 42.4
ALT or GPT (IU/L): 10.0
AST or GOT (IU/L): 20.0
Total Cholesterol (mmol/L): 5.4
Creatinine (mmol/L): 106.0
Gamma-glutamil transpeptidase (IU/L): 20.0
Glucose (mmol/L): 5.7
Hemoglobin (g/L): 132.0
HDL (mmol/L): 1.29
Potassium (mEq/L): 4.6
LDL (mmol/L): 3.36
Sodium (mEq/L): 141.0
Pro-BNP (ng/L): 1834.0
Protein (g/L): 69.0
T3 (pg/dL): 0.05
T4 (ng/L): 15.0
Troponin (ng/mL): 0.01
TSH (mIU/L): 3.02
Urea (mg/dL): 7.12
LVEF (%): 35.0
Medications: Beta Blockers, Digoxin, Loop Diuretics, ACE Inhibitor
ECG Impression:
- Ventricular Extrasystole: Polymorphic
        - Ventricular Tachycardia: Non-sustained VT
        - Non-sustained ventricular tachycardia (CH>10): Yes
        - Paroxysmal supraventricular tachyarrhythmia: Unknown paroxysmal supraventricular tachyarrhythmia code
        - Bradycardia: Unknown bradycardia code



In [46]:
# Save results
df_prompts.to_csv("../../../../local3/sswee/music_download/physionet.org/files/music-sudden-cardiac-death/1.0.1/subject-info-cleaned-with-prompts.csv", index = False)